# 02 - Data Cleaning and Feature Engineering

## Fuel Efficiency Prediction Project

In this notebook, we prepare the Auto MPG dataset for machine learning.

Raw data is usually not ready for model training. Before building a model, we need to:

- Check and handle missing values
- Remove unnecessary columns
- Convert categorical values into useful format
- Create new useful features
- Save the cleaned dataset

The cleaned dataset will be used in the next notebook for model building.

In [3]:
# Import required libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Show all columns clearly
pd.set_option("display.max_columns", None)

## 1. Load the Raw Dataset

We load the original `auto_mpg.csv` file from the `data` folder.

This dataset still contains raw values, so we need to clean it before using it for machine learning.

In [4]:
# Load the raw dataset

df = pd.read_csv("../data/auto_mpg.csv")

# Display first 5 rows
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,car_name
0,18.0,8,307.0,130.0,3504.0,12.0,70,1,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693.0,11.5,70,1,buick skylark 320
2,18.0,8,318.0,150.0,3436.0,11.0,70,1,plymouth satellite
3,16.0,8,304.0,150.0,3433.0,12.0,70,1,amc rebel sst
4,17.0,8,302.0,140.0,3449.0,10.5,70,1,ford torino


## 2. Check Dataset Information

Before cleaning, we check the dataset structure again.

This helps us understand:

- Number of rows and columns
- Column names
- Data types
- Missing values

In [5]:
# Check dataset shape

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 398
Columns: 9


In [6]:
# Check column information

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    float64
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    int64  
 8   car_name      398 non-null    str    
dtypes: float64(5), int64(3), str(1)
memory usage: 28.1 KB


## 3. Check Missing Values

Missing values can reduce model performance.

So, we check whether any column has missing values.

In [7]:
# Count missing values in each column

df.isnull().sum()

mpg             0
cylinders       0
displacement    0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
car_name        0
dtype: int64

### Missing Value Observation

The `horsepower` column may contain missing values.

Since horsepower is an important feature for predicting MPG, we should not remove the whole column.

Instead, we fill missing horsepower values using the **median** value.

Median is a good choice because it is less affected by extreme values than the mean.

In [8]:
# Check missing horsepower values before cleaning

print("Missing horsepower values before cleaning:")
print(df["horsepower"].isnull().sum())

Missing horsepower values before cleaning:
6


In [9]:
# Fill missing horsepower values using median

horsepower_median = df["horsepower"].median()

df["horsepower"] = df["horsepower"].fillna(horsepower_median)

print("Missing horsepower values after cleaning:")
print(df["horsepower"].isnull().sum())

Missing horsepower values after cleaning:
0


## 4. Drop Unnecessary Column

The `car_name` column contains the car name or model name.

For this project, we remove this column because:

- It is text data
- It is not useful for simple beginner-level regression
- It may make the model memorize specific car names instead of learning general patterns

So, we drop `car_name`.

In [10]:
# Drop car_name column if it exists

df = df.drop(columns=["car_name"], errors="ignore")

# Check remaining columns
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,18.0,8,307.0,130.0,3504.0,12.0,70,1
1,15.0,8,350.0,165.0,3693.0,11.5,70,1
2,18.0,8,318.0,150.0,3436.0,11.0,70,1
3,16.0,8,304.0,150.0,3433.0,12.0,70,1
4,17.0,8,302.0,140.0,3449.0,10.5,70,1


## 5. Convert Origin Codes into Region Names

The `origin` column is currently stored as numbers.

The meaning is:

- 1 = USA
- 2 = Europe
- 3 = Japan

To make the dataset easier to understand, we convert these numbers into readable names.

In [11]:
# Convert origin numbers into region names

df["origin"] = df["origin"].map({
    1: "USA",
    2: "Europe",
    3: "Japan"
})

# Check result
df["origin"].value_counts()

origin
USA       249
Japan      79
Europe     70
Name: count, dtype: int64

## 6. Feature Engineering

Feature engineering means creating new useful columns from existing columns.

Good features can help machine learning models make better predictions.

In this project, we create these new features:

1. `power_to_weight`
2. `displacement_per_cyl`
3. `is_v8`
4. `decade`

These features give more meaningful information about car performance and design.

### 6.1 Create Power-to-Weight Ratio

`power_to_weight` shows how much horsepower a car has compared to its weight.

Formula:

`power_to_weight = horsepower / weight * 1000`

This can be useful because a car's fuel efficiency depends not only on horsepower, but also on how heavy the car is.

In [12]:
# Create power-to-weight ratio feature

df["power_to_weight"] = (df["horsepower"] / df["weight"]) * 1000

df[["horsepower", "weight", "power_to_weight"]].head()

,horsepower,weight,power_to_weight
0,130.0,3504.0,37.100457
1,165.0,3693.0,44.679123
2,150.0,3436.0,43.655413
3,150.0,3433.0,43.693562
4,140.0,3449.0,40.591476


### 6.2 Create Displacement per Cylinder

`displacement_per_cyl` shows engine displacement per cylinder.

Formula:

`displacement_per_cyl = displacement / cylinders`

This helps us understand engine size more clearly.

In [13]:
# Create displacement per cylinder feature

df["displacement_per_cyl"] = df["displacement"] / df["cylinders"]

df[["displacement", "cylinders", "displacement_per_cyl"]].head()

,displacement,cylinders,displacement_per_cyl
0,307.0,8,38.375
1,350.0,8,43.750
2,318.0,8,39.750
3,304.0,8,38.000
4,302.0,8,37.750


### 6.3 Create V8 Engine Indicator

V8 cars usually have 8 cylinders.

We create a new column called `is_v8`.

- `1` means the car has 8 cylinders
- `0` means the car does not have 8 cylinders

This helps the model identify high-engine-power cars more clearly.

In [14]:
# Create V8 indicator feature

df["is_v8"] = (df["cylinders"] == 8).astype(int)

df[["cylinders", "is_v8"]].head(10)

,cylinders,is_v8
0,8,1
1,8,1
2,8,1
3,8,1
4,8,1
5,8,1
6,8,1
7,8,1
8,8,1
9,8,1


### 6.4 Create Decade Feature

The `model_year` column contains values like 70, 71, 72, and so on.

We create a `decade` feature to group cars by decade.

Example:

- 70, 71, 72 → 70s
- 80, 81, 82 → 80s

This helps identify fuel efficiency improvements over time.

In [15]:
# Create decade feature

df["decade"] = (df["model_year"] // 10) * 10

df[["model_year", "decade"]].head()

,model_year,decade
0,70,70
1,70,70
2,70,70
3,70,70
4,70,70


## 7. Convert Categorical Column Using One-Hot Encoding

Machine learning models cannot directly understand text categories like:

- USA
- Europe
- Japan

So we convert the `origin` column into numeric columns using one-hot encoding.

We use `drop_first=True` to avoid duplicate information.

In [16]:
# Convert origin column into numeric dummy columns

df_encoded = pd.get_dummies(df, columns=["origin"], drop_first=True)

# Convert True/False values into 1/0 if needed
bool_cols = df_encoded.select_dtypes(include="bool").columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

df_encoded.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,power_to_weight,displacement_per_cyl,is_v8,decade,origin_Japan,origin_USA
0,18.0,8,307.0,130.0,3504.0,12.0,70,37.100457,38.375,1,70,0,1
1,15.0,8,350.0,165.0,3693.0,11.5,70,44.679123,43.750,1,70,0,1
2,18.0,8,318.0,150.0,3436.0,11.0,70,43.655413,39.750,1,70,0,1
3,16.0,8,304.0,150.0,3433.0,12.0,70,43.693562,38.000,1,70,0,1
4,17.0,8,302.0,140.0,3449.0,10.5,70,40.591476,37.750,1,70,0,1


## 8. Check Cleaned Dataset

Now we check the final cleaned dataset.

At this point:

- Missing values are handled
- Unnecessary text column is removed
- New features are created
- Categorical values are encoded

In [17]:
# Check final dataset information

df_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   mpg                   398 non-null    float64
 1   cylinders             398 non-null    int64  
 2   displacement          398 non-null    float64
 3   horsepower            398 non-null    float64
 4   weight                398 non-null    float64
 5   acceleration          398 non-null    float64
 6   model_year            398 non-null    int64  
 7   power_to_weight       398 non-null    float64
 8   displacement_per_cyl  398 non-null    float64
 9   is_v8                 398 non-null    int64  
 10  decade                398 non-null    int64  
 11  origin_Japan          398 non-null    int64  
 12  origin_USA            398 non-null    int64  
dtypes: float64(7), int64(6)
memory usage: 40.6 KB


## 9. Final Cleaned Dataset Preview

Now we display the first few rows of the cleaned dataset.

In [18]:
# Display cleaned dataset

df_encoded.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,power_to_weight,displacement_per_cyl,is_v8,decade,origin_Japan,origin_USA
0,18.0,8,307.0,130.0,3504.0,12.0,70,37.100457,38.375,1,70,0,1
1,15.0,8,350.0,165.0,3693.0,11.5,70,44.679123,43.750,1,70,0,1
2,18.0,8,318.0,150.0,3436.0,11.0,70,43.655413,39.750,1,70,0,1
3,16.0,8,304.0,150.0,3433.0,12.0,70,43.693562,38.000,1,70,0,1
4,17.0,8,302.0,140.0,3449.0,10.5,70,40.591476,37.750,1,70,0,1


## 10. Save the Cleaned Dataset

Finally, we save the cleaned dataset as:

`auto_mpg_cleaned.csv`

This file will be used in the model building notebook.

In [19]:
# Save cleaned dataset to data folder

df_encoded.to_csv("../data/auto_mpg_cleaned.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


## 11. Data Cleaning Summary

In this notebook, we completed the data cleaning and feature engineering process.

Main tasks completed:

1. Loaded the raw Auto MPG dataset
2. Checked dataset shape and information
3. Checked missing values
4. Filled missing `horsepower` values using median
5. Removed the `car_name` column
6. Converted `origin` codes into readable region names
7. Created new useful features:
   - `power_to_weight`
   - `displacement_per_cyl`
   - `is_v8`
   - `decade`
8. Converted categorical values using one-hot encoding
9. Saved the cleaned dataset as `auto_mpg_cleaned.csv`

The dataset is now ready for machine learning model training.